# Build: `adif__mainDataTable_notebook` — V2 (Single-Pass UNION CTE)

**Purpose:** Replace two-section notebook (CREATE + INSERT) with a single-pass UNION CTE that
preserves source lineage for every metric.

**Target table (shadow):** `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook_v2_test`

**Production table (baseline for validation):** `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`

## Key changes from V1

| Change | V1 (two-section) | V2 (single-pass) |
|--------|-------------------|-------------------|
| Social spend | → `final_spend` + `d_daily_recalculated_cost` (DCM column!) | → `s_spend` only |
| Social impressions | → `final_impressions` + `d_impressions` (DCM final-impression column) | → `s_impressions` only |
| Social video | → `d_video_plays`, `d_video_comps` (DCM columns!) | → `s_video_plays`, `s_video_comps` |
| Planned spend | Single `planned_daily_spend_pk` (Prisma + pacing mixed) | `prisma_planned_spend` + `pacing_planned_spend` → COALESCE |
| `final_spend` | Hardcoded per INSERT branch | COALESCE(updated_fpd, fpd, dcm, social) after UNION |
| `data_source_primary` | Hardcoded per INSERT branch | Derived from source columns |

## Structure

```
BLOCK 1: digital_rows   — SELECT explicit cols from updated_fpd_view + s_*=NULL, pacing_planned_spend=NULL
BLOCK 2: social chain   — social_daily → tokens → enriched → pacing_dedup → pacing_daily
                          → social_with_plan (per-row calcs) → social_mapped (window rollups)
BLOCK 3: unioned        — SELECT * FROM digital_rows UNION ALL SELECT * FROM social_mapped
BLOCK 4: with_rollups   — COALESCE canonical cols + package window rollups
BLOCK 5: final SELECT   — SELECT * FROM with_rollups
```

## Key sources
- `repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view` — digital rows (Prisma + DCM + FPD)
- `repo_stg.stg__adif__social_crossplatform` — normalized social actuals
- `repo_int.crossplatform_pacing` — planned spend by campaign / ad_group

## Validation
Run `sql/test__adif__social_mapping_v2_vs_current.sql` sections VAL-01 through VAL-07
to compare shadow table vs production baseline.

## Setup

In [1]:
# @title Project & region config
PROJECT_ID = "looker-studio-pro-452620"  # @param {type:"string"}
REGION     = "US"                         # @param {type:"string"}

print(f"Project : {PROJECT_ID}")
print(f"Region  : {REGION}")

Project : looker-studio-pro-452620
Region  : US


In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

ModuleNotFoundError: No module named 'google.colab'

---
## SINGLE-PASS BUILD

Creates `adif__mainDataTable_notebook_v2_test` (shadow table).  
**Production table is not touched** — run validation queries first.

### Column contract (unioned schema — 71 columns)
Both `digital_rows` and `social_mapped` emit exactly these columns.  
Canonical columns (`final_spend`, `planned_daily_spend_pk`, etc.) are computed **after** the UNION in `with_rollups`.

| Group | Columns |
|-------|--------|
| Core IDs | `package_id_joined`, `date`, `flight_status_flag` |
| DCM source | `d_media_cost`, `d_impressions`, `d_daily_recalculated_cost`, `d_daily_recalculated_imps`, `d_clicks`, `d_video_plays`, `d_video_comps`, `d_min_date`, `d_max_date`, `d_min_flight_date`, `d_max_flight_date` |
| FPD source | `fpd_spend`, `fpd_impressions`, `updated_fpd_spend`, `updated_fpd_impressions`, `updated_fpd_suppliers`, `updated_fpd_initiatives`, `updated_fpd_data_timestamp`, `pkg_updated_fpd_spend`, `pkg_updated_fpd_impressions` |
| Social source (NEW) | `s_spend`, `s_impressions`, `s_clicks`, `s_video_plays`, `s_video_comps`, `s_video_views` |
| Planned source (NEW) | `prisma_planned_spend`, `prisma_planned_impressions`, `prisma_planned_clicks`, `pacing_planned_spend` |
| Dimensions | `package_type`, `cost_method`, `buy_type`, `buy_category`, `advertiser_name`, `campaign_name`, `supplier_code`, `supplier_name`, `channel_if_buy_category_custom_45`, `package_id`, `package_name`, `PlacementName`, `placement_name`, `placement_id`, `package_group_name`, `start_date`, `end_date`, `report_date`, `script_run_date`, `channel`, `channel_raw`, `channel_group`, `campaign_friendly`, `p_package_friendly`, `placement_type_site`, `line_item_name`, `media_type`, `funnel`, `kpi`, `initiative`, `initative`, `n_of_placements`, `planned_imps_pk`, `planned_cost_pk` |

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- @title: Build adif__mainDataTable_notebook — V2: Single-Pass UNION CTE
-- @description:
--   Single CREATE OR REPLACE TABLE replacing two-section notebook (Section 1 + Section 2).
--   Bug fix: social rows no longer write spend/impressions/video into DCM columns.
--   All 130 original columns preserved. 10 new source-prefix columns added.
--
--   Column contract: digital_rows and social_mapped emit identical 140-column schemas.
--   Canonical columns (final_spend, planned_daily_spend_pk, data_source_primary, etc.)
--   are recomputed in with_rollups after the UNION.
--
-- @inputs:
--   repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view
--   repo_stg.stg__adif__social_crossplatform
--   repo_int.crossplatform_pacing
-- @outputs:
--   repo_stg.adif__mainDataTable_notebook_v2_test  (shadow — run VAL-01..07 before promoting)
-- @last_updated: 2026-03-11
-- @schema-verified-against: adif__mainDataTable_notebook (130 cols, confirmed 2026-03-04)

CREATE OR REPLACE TABLE `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook_v2_test` AS
WITH

-- ════════════════════════════════════════════════════════════════════════════
-- BLOCK 1: DIGITAL ROWS  (130 view cols + 10 new source-prefix cols = 140 total)
--   SELECT * keeps all 130 original columns intact.
--   New column names (prisma_planned_spend etc.) do not conflict with existing names.
--   Social source columns are NULL — digital rows have no social data.
-- ════════════════════════════════════════════════════════════════════════════
digital_rows AS (
  SELECT
    *,
    -- ── Planned source prefix columns (NEW) ─────────────────────────────────
    planned_daily_spend_pk          AS prisma_planned_spend,
    planned_daily_impressions_pk    AS prisma_planned_impressions,
    planned_clicks                  AS prisma_planned_clicks,
    -- ── Social source columns: NULL for digital rows ─────────────────────────
    CAST(NULL AS FLOAT64) AS pacing_planned_spend,
    CAST(NULL AS FLOAT64) AS s_spend,
    CAST(NULL AS FLOAT64) AS s_impressions,
    CAST(NULL AS FLOAT64) AS s_clicks,
    CAST(NULL AS FLOAT64) AS s_video_plays,
    CAST(NULL AS FLOAT64) AS s_video_comps,
    CAST(NULL AS FLOAT64) AS s_video_views
  FROM `looker-studio-pro-452620.repo_stg.adif__prisma_expanded_plus_dcm_updated_fpd_view`
),

-- ════════════════════════════════════════════════════════════════════════════
-- BLOCK 2: SOCIAL CTE CHAIN
-- ════════════════════════════════════════════════════════════════════════════

-- [2.1] Normalize raw social to ad-level daily grain
social_daily AS (
  SELECT
    date_day,
    CASE
      WHEN LOWER(social_platform) IN ('facebook_ads', 'instagram_ads', 'meta') THEN 'meta'
      WHEN LOWER(social_platform) IN ('tiktok_ads', 'tiktok_ads_adif', 'tiktok')  THEN 'tiktok'
      ELSE LOWER(social_platform)
    END AS social_platform_norm,
    CAST(campaign_id AS STRING) AS campaign_id,
    campaign_name,
    CAST(ad_group_id AS STRING) AS ad_group_id,
    ad_group_name,
    CAST(ad_id AS STRING)       AS ad_id,
    ad_name,
    account_name,
    SUM(spend)             AS spend,
    SUM(impressions)       AS impressions,
    SUM(clicks)            AS clicks,
    SUM(video_play)        AS video_play,
    SUM(video_view)        AS video_view,
    SUM(video_views_p_100) AS video_complete_proxy
  FROM `looker-studio-pro-452620.repo_stg.stg__adif__social_crossplatform`
  GROUP BY 1,2,3,4,5,6,7,8,9
),

-- [2.2] Parse token-based dimensions
social_tokens AS (
  SELECT
    s.*,
    SPLIT(s.ad_group_name, '_')[SAFE_OFFSET(1)] AS initiative_token_ag,
    SPLIT(s.ad_group_name, '_')[SAFE_OFFSET(2)] AS funnel_token_ag,
    LOWER(REGEXP_EXTRACT(s.ad_group_name,
      r'(?i)_(facebook|instagram|tiktok|youtube|snapchat|linkedin|pinterest)(?:_|$)'
    )) AS supplier_token_ag,
    SPLIT(s.ad_name, '_')[SAFE_OFFSET(1)] AS initiative_token_ad,
    SPLIT(s.ad_name, '_')[SAFE_OFFSET(2)] AS line_item_name_token_ad,
    SPLIT(s.ad_name, '_')[SAFE_OFFSET(3)] AS placement_type_token_1_ad,
    SPLIT(s.ad_name, '_')[SAFE_OFFSET(4)] AS buy_category_token_1_ad
  FROM social_daily AS s
),

-- [2.3] Resolve dimensions + allocation helpers
social_enriched AS (
  SELECT
    t.*,
    COALESCE(t.initiative_token_ag, t.initiative_token_ad, 'NA') AS initiative,
    COALESCE(t.line_item_name_token_ad, 'NA')                    AS line_item_name,
    CONCAT(
      COALESCE(t.placement_type_token_1_ad, 'na'), '_',
      COALESCE(t.buy_category_token_1_ad, 'na')
    ) AS placement_type_buy_category,
    CASE
      WHEN t.supplier_token_ag = 'facebook'     THEN 'FB'
      WHEN t.supplier_token_ag = 'instagram'    THEN 'IG'
      WHEN t.supplier_token_ag = 'tiktok'       THEN 'TT'
      WHEN t.supplier_token_ag = 'pinterest'    THEN 'PT'
      WHEN t.social_platform_norm = 'meta'      THEN 'FB'
      WHEN t.social_platform_norm = 'tiktok'    THEN 'TT'
      WHEN t.social_platform_norm = 'pinterest' THEN 'PT'
      ELSE UPPER(SUBSTR(t.social_platform_norm, 1, 2))
    END AS supplier_code_short,
    CASE
      WHEN t.supplier_token_ag = 'facebook'     THEN 'Facebook'
      WHEN t.supplier_token_ag = 'instagram'    THEN 'Instagram'
      WHEN t.supplier_token_ag = 'tiktok'       THEN 'TikTok'
      WHEN t.supplier_token_ag = 'pinterest'    THEN 'Pinterest'
      WHEN t.social_platform_norm = 'meta'      THEN 'Facebook'
      WHEN t.social_platform_norm = 'tiktok'    THEN 'TikTok'
      WHEN t.social_platform_norm = 'pinterest' THEN 'Pinterest'
      ELSE INITCAP(t.social_platform_norm)
    END AS supplier_name_full,
    SUM(t.spend) OVER (
      PARTITION BY t.date_day, t.social_platform_norm, t.campaign_id, t.ad_group_id
    ) AS ad_group_day_spend,
    COUNT(*) OVER (
      PARTITION BY t.date_day, t.social_platform_norm, t.campaign_id, t.ad_group_id
    ) AS ad_group_day_ad_count
  FROM social_tokens AS t
),

-- [2.4] Dedup pacing budgets
pacing_dedup AS (
  SELECT
    CASE
      WHEN LOWER(platform) IN ('facebook', 'instagram', 'meta') THEN 'meta'
      WHEN LOWER(platform) = 'tiktok'                           THEN 'tiktok'
      ELSE LOWER(platform)
    END AS social_platform_norm,
    CAST(c_id  AS STRING) AS campaign_id,
    CAST(ag_id AS STRING) AS ad_group_id,
    ag_start_date, ag_end_date,
    MAX(final_budget) AS final_budget
  FROM `looker-studio-pro-452620.repo_int.crossplatform_pacing`
  WHERE REGEXP_CONTAINS(IFNULL(campaign_name, ''), r'WP_')
    AND final_budget > 0 AND ag_start_date IS NOT NULL AND ag_end_date IS NOT NULL
  GROUP BY 1,2,3,4,5
),

-- [2.5] Daily pacing
pacing_daily AS (
  SELECT
    day AS date_day, social_platform_norm, campaign_id, ad_group_id,
    SUM(final_budget / NULLIF(DATE_DIFF(ag_end_date, ag_start_date, DAY) + 1, 0)) AS planned_daily_spend
  FROM pacing_dedup,
    UNNEST(GENERATE_DATE_ARRAY(ag_start_date, ag_end_date)) AS day
  GROUP BY 1,2,3,4
),

-- [2.6] Join social to pacing + compute per-row pacing allocation as a real column
--   (stored as column so window functions in social_mapped can reference it directly)
social_with_plan AS (
  SELECT
    e.*,
    p.planned_daily_spend,
    CASE
      WHEN p.planned_daily_spend IS NULL   THEN NULL
      WHEN e.ad_group_day_spend > 0        THEN p.planned_daily_spend * SAFE_DIVIDE(e.spend, e.ad_group_day_spend)
      WHEN e.ad_group_day_ad_count > 0     THEN p.planned_daily_spend / e.ad_group_day_ad_count
      ELSE NULL
    END AS pacing_planned_spend
  FROM social_enriched AS e
  LEFT JOIN pacing_daily AS p
    ON  e.date_day = p.date_day AND e.social_platform_norm = p.social_platform_norm
    AND e.campaign_id = p.campaign_id AND e.ad_group_id = p.ad_group_id
),

-- [2.7] Map social rows to full 140-column ADIF schema
--   Column ORDER must exactly match digital_rows (view col order 1-130, then new cols 131-140).
--   KEY BUG FIX: d_daily_recalculated_cost, d_video_plays, d_video_comps are NULL.
--   spend → s_spend (not final_spend, not d_daily_recalculated_cost).
--   pacing_planned_spend → pacing_planned_spend (not planned_daily_spend_pk).
social_mapped AS (
  SELECT
    -- ── Original view columns 1–130, in schema order ─────────────────────────
    -- 1  package_id_joined STRING
    CONCAT('Social_', supplier_code_short, '_', initiative, '_', line_item_name) AS package_id_joined,
    -- 2  date DATE
    date_day                                                           AS date,
    -- 3  flight_status_flag STRING
    CASE
      WHEN CURRENT_DATE() > MAX(date_day) OVER (PARTITION BY ad_group_id) THEN 'ended'
      ELSE 'live'
    END                                                                AS flight_status_flag,
    -- 4  d_daily_recalculated_cost FLOAT  — BUG FIX: was spend
    CAST(NULL AS FLOAT64)                                              AS d_daily_recalculated_cost,
    -- 5  d_daily_recalculated_imps INTEGER — BUG FIX: was CAST(impressions AS INT64)
    CAST(NULL AS INT64)                                                AS d_daily_recalculated_imps,
    -- 6  d_impressions INTEGER
    CAST(NULL AS INT64)                                                AS d_impressions,
    -- 7  d_media_cost FLOAT
    CAST(NULL AS FLOAT64)                                              AS d_media_cost,
    -- 8  d_clicks INTEGER — BUG FIX: was CAST(clicks AS INT64)
    CAST(NULL AS INT64)                                                AS d_clicks,
    -- 9  d_video_plays INTEGER — BUG FIX: was SAFE_CAST(ROUND(video_play) AS INT64)
    CAST(NULL AS INT64)                                                AS d_video_plays,
    -- 10 d_video_comps INTEGER — BUG FIX: was SAFE_CAST(ROUND(video_complete_proxy) AS INT64)
    CAST(NULL AS INT64)                                                AS d_video_comps,
    -- 11 d_min_date DATE
    MIN(date_day) OVER (PARTITION BY ad_group_id)                      AS d_min_date,
    -- 12 d_max_date DATE
    MAX(date_day) OVER (PARTITION BY ad_group_id)                      AS d_max_date,
    -- 13 d_prorated_planned_cost_pk FLOAT
    CAST(NULL AS FLOAT64)                                              AS d_prorated_planned_cost_pk,
    -- 14 d_prorated_planned_imps_pk FLOAT
    CAST(NULL AS FLOAT64)                                              AS d_prorated_planned_imps_pk,
    -- 15 d_min_flight_date DATE
    MIN(date_day) OVER (PARTITION BY ad_group_id)                      AS d_min_flight_date,
    -- 16 d_max_flight_date DATE
    MAX(date_day) OVER (PARTITION BY ad_group_id)                      AS d_max_flight_date,
    -- 17 d_daily_cpm FLOAT
    CASE
      WHEN impressions > 0 THEN spend * 1000.0 / impressions
      ELSE NULL
    END                                                                AS d_daily_cpm,
    -- 18 d_total_delivered_imps INTEGER
    SAFE_CAST(ROUND(
      SUM(impressions) OVER (PARTITION BY ad_group_id)
    ) AS INT64)                                                        AS d_total_delivered_imps,
    -- 19 d_total_del_inflight_imps INTEGER
    SAFE_CAST(ROUND(
      SUM(impressions) OVER (PARTITION BY ad_group_id)
    ) AS INT64)                                                        AS d_total_del_inflight_imps,
    -- 20 planned_daily_spend_pk FLOAT (placeholder — excepted + recomputed in with_rollups)
    pacing_planned_spend                                               AS planned_daily_spend_pk,
    -- 21 planned_daily_impressions_pk FLOAT
    CAST(NULL AS FLOAT64)                                              AS planned_daily_impressions_pk,
    -- 22 max_prismaEXP_report_date DATE
    CAST(NULL AS DATE)                                                 AS max_prismaEXP_report_date,
    -- 23 final_clicks FLOAT (placeholder — excepted + recomputed in with_rollups)
    CAST(clicks AS FLOAT64)                                            AS final_clicks,
    -- 24 final_sends INTEGER
    CAST(NULL AS INT64)                                                AS final_sends,
    -- 25 final_opens INTEGER
    CAST(NULL AS INT64)                                                AS final_opens,
    -- 26 pkg_est_spend FLOAT
    SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id)          AS pkg_est_spend,
    -- 27 pkg_est_imps FLOAT
    CAST(NULL AS FLOAT64)                                              AS pkg_est_imps,
    -- 28 pkg_over_bool BOOLEAN
    CASE
      WHEN SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id) IS NULL
        OR SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id) = 0 THEN NULL
      ELSE SUM(spend) OVER (PARTITION BY ad_group_id)
           > SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id)
    END                                                                AS pkg_over_bool,
    -- 29 pkg_over_flag INTEGER
    CASE
      WHEN SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id) IS NULL
        OR SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id) = 0 THEN NULL
      WHEN SUM(spend) OVER (PARTITION BY ad_group_id)
           > SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id) THEN 1
      ELSE 0
    END                                                                AS pkg_over_flag,
    -- 30 package_type STRING
    'Standalone'                                                      AS package_type,
    -- 31 placement_type STRING
    CAST(NULL AS STRING)                                               AS placement_type,
    -- 32 placement_name STRING
    ad_name                                                            AS placement_name,
    -- 33 start_date DATE
    MIN(date_day) OVER (PARTITION BY ad_group_id)                      AS start_date,
    -- 34 end_date DATE
    MAX(date_day) OVER (PARTITION BY ad_group_id)                      AS end_date,
    -- 35 cost_method STRING
    CAST(NULL AS STRING)                                               AS cost_method,
    -- 36 buy_type STRING
    'social'                                                          AS buy_type,
    -- 37 buy_category STRING
    placement_type_buy_category                                        AS buy_category,
    -- 38 click_through_url STRING
    CAST(NULL AS STRING)                                               AS click_through_url,
    -- 39 advertiser_name STRING
    'Forevermark US'                                                  AS advertiser_name,
    -- 40 campaign_name STRING
    campaign_name,
    -- 41 supplier_code STRING
    supplier_code_short                                                AS supplier_code,
    -- 42 supplier_name STRING
    supplier_name_full                                                 AS supplier_name,
    -- 43 planned_actions INTEGER
    CAST(NULL AS INT64)                                                AS planned_actions,
    -- 44 planned_amount FLOAT
    CAST(NULL AS FLOAT64)                                              AS planned_amount,
    -- 45 planned_clicks INTEGER
    CAST(NULL AS INT64)                                                AS planned_clicks,
    -- 46 planned_impressions INTEGER
    CAST(NULL AS INT64)                                                AS planned_impressions,
    -- 47 planned_units INTEGER
    CAST(NULL AS INT64)                                                AS planned_units,
    -- 48 unit_type STRING
    CAST(NULL AS STRING)                                               AS unit_type,
    -- 49 dimension STRING
    CAST(NULL AS STRING)                                               AS dimension,
    -- 50 media_name STRING
    CAST(NULL AS STRING)                                               AS media_name,
    -- 51 product_code STRING
    CAST(NULL AS STRING)                                               AS product_code,
    -- 52 product_name STRING
    CAST(NULL AS STRING)                                               AS product_name,
    -- 53 external_entity_id STRING
    CAST(NULL AS STRING)                                               AS external_entity_id,
    -- 54 provider_name STRING
    CAST(NULL AS STRING)                                               AS provider_name,
    -- 55 package_group_name STRING
    ad_group_name                                                      AS package_group_name,
    -- 56 out_of_home_end_date STRING
    CAST(NULL AS STRING)                                               AS out_of_home_end_date,
    -- 57 placement_cap_cost STRING
    CAST(NULL AS STRING)                                               AS placement_cap_cost,
    -- 58 served_by STRING
    CAST(NULL AS STRING)                                               AS served_by,
    -- 59 payable_rate FLOAT
    CAST(NULL AS FLOAT64)                                              AS payable_rate,
    -- 60 channel_if_buy_category_custom_45 STRING
    CONCAT('social_', LOWER(placement_type_buy_category))            AS channel_if_buy_category_custom_45,
    -- 61 placement_type_site STRING
    placement_type_buy_category                                        AS placement_type_site,
    -- 62 device_type STRING
    CAST(NULL AS STRING)                                               AS device_type,
    -- 63 audience_type STRING
    CAST(NULL AS STRING)                                               AS audience_type,
    -- 64 audience STRING
    CAST(NULL AS STRING)                                               AS audience,
    -- 65 initative STRING (intentional typo — matches original schema)
    initiative                                                         AS initative,
    -- 66 media_type STRING
    CASE
      WHEN COALESCE(video_play, 0) > 0 OR COALESCE(video_complete_proxy, 0) > 0    THEN 'Video'
      WHEN REGEXP_CONTAINS(ad_group_name, r'(?i)_VV(_|$)|_Video(_|$)|_Reel(_|$)') THEN 'Video'
      WHEN REGEXP_CONTAINS(ad_name, r'(?i)video|reel')                             THEN 'Video'
      ELSE 'Static'
    END                                                                AS media_type,
    -- 67 line_item_name STRING
    line_item_name,
    -- 68 ad_size_asset STRING
    CAST(NULL AS STRING)                                               AS ad_size_asset,
    -- 69 funnel STRING
    CASE
      WHEN funnel_token_ag IN ('VV', 'Reach') THEN 'Awareness'
      WHEN funnel_token_ag = 'Engagement'      THEN 'Consideration'
      ELSE 'Awareness'
    END                                                                AS funnel,
    -- 70 kpi STRING
    CASE
      WHEN funnel_token_ag IN ('VV', 'VideoViews', 'Video') THEN 'Video Views'
      WHEN funnel_token_ag = 'Reach'                           THEN 'Impressions'
      WHEN funnel_token_ag = 'Engagement'                      THEN 'Engagement'
      ELSE 'Impressions'
    END                                                                AS kpi,
    -- 71 geo_market STRING
    CAST(NULL AS STRING)                                               AS geo_market,
    -- 72 ad_server_placement_id INTEGER
    CAST(NULL AS INT64)                                                AS ad_server_placement_id,
    -- 73 placement_id STRING
    CAST(ad_id AS STRING)                                              AS placement_id,
    -- 74 placement_comments STRING
    CAST(NULL AS STRING)                                               AS placement_comments,
    -- 75 days_in_out_of_home_end_date DATE
    CAST(NULL AS DATE)                                                 AS days_in_out_of_home_end_date,
    -- 76 package_id STRING
    CAST(ad_group_id AS STRING)                                        AS package_id,
    -- 77 package_name STRING
    ad_group_name                                                      AS package_name,
    -- 78 PlacementName STRING
    ad_name                                                            AS PlacementName,
    -- 79 report_date DATE
    CURRENT_DATE()                                                     AS report_date,
    -- 80 script_run_date TIMESTAMP
    CURRENT_TIMESTAMP()                                                AS script_run_date,
    -- 81 channel STRING
    CASE
      WHEN COALESCE(video_play, 0) > 0 OR COALESCE(video_complete_proxy, 0) > 0    THEN 'social_paid video'
      WHEN REGEXP_CONTAINS(ad_group_name, r'(?i)_VV(_|$)|_Video(_|$)|_Reel(_|$)') THEN 'social_paid video'
      WHEN REGEXP_CONTAINS(ad_name, r'(?i)video|reel')                             THEN 'social_paid video'
      ELSE 'social_paid display'
    END                                                                AS channel,
    -- 82 channel_raw STRING
    'social'                                                          AS channel_raw,
    -- 83 channel_group STRING
    'social'                                                          AS channel_group,
    -- 84 campaign_friendly STRING
    campaign_name                                                      AS campaign_friendly,
    -- 85 p_package_friendly STRING
    CONCAT('Social_', supplier_code_short, '_', initiative, '_', line_item_name) AS p_package_friendly,
    -- 86 n_of_placements INTEGER
    1                                                                  AS n_of_placements,
    -- 87 planned_imps_pk INTEGER
    CAST(NULL AS INT64)                                                AS planned_imps_pk,
    -- 88 planned_cost_pk FLOAT
    SUM(pacing_planned_spend) OVER (PARTITION BY ad_group_id)          AS planned_cost_pk,
    -- 89 total_days INTEGER
    CAST(NULL AS INT64)                                                AS total_days,
    -- 90 daily_spend FLOAT
    CAST(NULL AS FLOAT64)                                              AS daily_spend,
    -- 91 daily_impressions FLOAT
    CAST(NULL AS FLOAT64)                                              AS daily_impressions,
    -- 92 channel_any_pkg_over INTEGER
    CAST(NULL AS INT64)                                                AS channel_any_pkg_over,
    -- 93 site_any_pkg_over INTEGER
    CAST(NULL AS INT64)                                                AS site_any_pkg_over,
    -- 94 gsMediaTeam_channel STRING
    CAST(NULL AS STRING)                                               AS gsMediaTeam_channel,
    -- 95 final_table_refresh_date TIMESTAMP
    CAST(NULL AS TIMESTAMP)                                            AS final_table_refresh_date,
    -- 96 initiative STRING
    initiative,
    -- 97–104 fpd_orig_* columns: NULL for social
    CAST(NULL AS FLOAT64)   AS fpd_orig_impressions,
    CAST(NULL AS FLOAT64)   AS fpd_orig_spend,
    CAST(NULL AS FLOAT64)   AS fpd_orig_clicks,
    CAST(NULL AS INT64)     AS fpd_orig_sends,
    CAST(NULL AS INT64)     AS fpd_orig_opens,
    CAST(NULL AS FLOAT64)   AS fpd_orig_benchmark,
    CAST(NULL AS INT64)     AS fpd_orig_benchmark_metric,
    CAST(NULL AS STRING)    AS fpd_orig_creative,
    -- 105–109 fpd_updated_* columns: NULL for social
    CAST(NULL AS FLOAT64)   AS fpd_updated_impressions,
    CAST(NULL AS FLOAT64)   AS fpd_updated_spend,
    CAST(NULL AS STRING)    AS fpd_updated_suppliers,
    CAST(NULL AS STRING)    AS fpd_updated_initiatives,
    CAST(NULL AS TIMESTAMP) AS fpd_updated_data_timestamp,
    -- 110–117 fpd_* combined columns: NULL for social
    CAST(NULL AS FLOAT64)   AS fpd_impressions,
    CAST(NULL AS FLOAT64)   AS fpd_spend,
    CAST(NULL AS FLOAT64)   AS fpd_clicks,
    CAST(NULL AS INT64)     AS fpd_sends,
    CAST(NULL AS INT64)     AS fpd_opens,
    CAST(NULL AS FLOAT64)   AS fpd_benchmark,
    CAST(NULL AS INT64)     AS fpd_benchmark_metric,
    CAST(NULL AS STRING)    AS fpd_creative,
    -- 118 final_impressions_new FLOAT (view artifact, NULL for social)
    CAST(NULL AS FLOAT64)   AS final_impressions_new,
    -- 119 final_spend_new FLOAT (view artifact, NULL for social)
    CAST(NULL AS FLOAT64)   AS final_spend_new,
    -- 120 data_source_primary STRING (placeholder — excepted + recomputed in with_rollups)
    social_platform_norm                                               AS data_source_primary,
    -- 121 final_impressions FLOAT (placeholder — excepted + recomputed in with_rollups)
    CAST(impressions AS FLOAT64)                                       AS final_impressions,
    -- 122 final_spend FLOAT (placeholder — excepted + recomputed in with_rollups)
    spend                                                              AS final_spend,
    -- 123 pkg_act_imps FLOAT
    SUM(CAST(impressions AS FLOAT64)) OVER (PARTITION BY ad_group_id)  AS pkg_act_imps,
    -- 124 pkg_act_spend FLOAT
    SUM(spend) OVER (PARTITION BY ad_group_id)                         AS pkg_act_spend,
    -- 125–130 pkg_fpd_* columns: NULL for social
    CAST(NULL AS FLOAT64)   AS pkg_fpd_orig_impressions,
    CAST(NULL AS FLOAT64)   AS pkg_fpd_orig_spend,
    CAST(NULL AS FLOAT64)   AS pkg_fpd_updated_impressions,
    CAST(NULL AS FLOAT64)   AS pkg_fpd_updated_spend,
    CAST(NULL AS FLOAT64)   AS pkg_fpd_combined_impressions,
    CAST(NULL AS FLOAT64)   AS pkg_fpd_combined_spend,

    -- ── NEW source prefix columns 131–140 ────────────────────────────────────
    CAST(NULL AS FLOAT64)   AS prisma_planned_spend,
    CAST(NULL AS FLOAT64)   AS prisma_planned_impressions,
    CAST(NULL AS INT64)     AS prisma_planned_clicks,
    -- pacing_planned_spend: real column from social_with_plan (not an alias here)
    pacing_planned_spend,
    -- Social source metrics (KEY CHANGE from v1: s_* not d_* or final_*)
    spend                                   AS s_spend,
    CAST(impressions AS FLOAT64)            AS s_impressions,
    CAST(clicks AS FLOAT64)                 AS s_clicks,
    CAST(video_play AS FLOAT64)             AS s_video_plays,
    CAST(video_complete_proxy AS FLOAT64)   AS s_video_comps,
    CAST(video_view AS FLOAT64)             AS s_video_views

  FROM social_with_plan
),

-- ════════════════════════════════════════════════════════════════════════════
-- BLOCK 3: UNION (schema contract enforced — both CTEs emit 140 columns in same order)
-- ════════════════════════════════════════════════════════════════════════════
unioned AS (
  SELECT * FROM digital_rows
  UNION ALL
  SELECT * FROM social_mapped
),

-- ════════════════════════════════════════════════════════════════════════════
-- BLOCK 4: CANONICAL COLUMNS
--   EXCEPT placeholders and recompute from source columns.
--   Source-aware CASE logic prevents zero-filled FPD placeholders from masking DCM rows.
--   DCM spend stays on `d_daily_recalculated_cost`; DCM final impressions use `d_impressions`.
-- ════════════════════════════════════════════════════════════════════════════
with_rollups AS (
  SELECT
    u.* EXCEPT(
      final_spend, final_impressions, final_clicks,
      planned_daily_spend_pk,
      data_source_primary
    ),

    -- ── Canonical delivery metrics ────────────────────────────────────────────
    CASE
      WHEN fpd_updated_spend IS NOT NULL OR fpd_updated_impressions IS NOT NULL
        OR fpd_orig_spend IS NOT NULL OR fpd_orig_impressions IS NOT NULL THEN fpd_spend
      WHEN d_daily_recalculated_cost IS NOT NULL
        OR d_daily_recalculated_imps IS NOT NULL THEN d_daily_recalculated_cost
      WHEN s_spend IS NOT NULL OR s_impressions IS NOT NULL THEN s_spend
      ELSE NULL
    END                                                                     AS final_spend,
    CASE
      WHEN fpd_updated_spend IS NOT NULL OR fpd_updated_impressions IS NOT NULL
        OR fpd_orig_spend IS NOT NULL OR fpd_orig_impressions IS NOT NULL THEN fpd_impressions
      WHEN d_daily_recalculated_cost IS NOT NULL
        OR d_daily_recalculated_imps IS NOT NULL THEN CAST(d_impressions AS FLOAT64)
      WHEN s_spend IS NOT NULL OR s_impressions IS NOT NULL THEN s_impressions
      ELSE NULL
    END                                                                     AS final_impressions,
    CASE
      WHEN fpd_updated_spend IS NOT NULL OR fpd_updated_impressions IS NOT NULL
        OR fpd_orig_spend IS NOT NULL OR fpd_orig_impressions IS NOT NULL THEN fpd_clicks
      WHEN d_daily_recalculated_cost IS NOT NULL
        OR d_daily_recalculated_imps IS NOT NULL THEN CAST(d_clicks AS FLOAT64)
      WHEN s_spend IS NOT NULL OR s_impressions IS NOT NULL THEN s_clicks
      ELSE NULL
    END                                                                     AS final_clicks,

    -- ── Planned (source-separated → canonical) ────────────────────────────────
    COALESCE(prisma_planned_spend, pacing_planned_spend)                    AS planned_daily_spend_pk,

    -- ── Data source label (derived from source columns, not hardcoded) ────────
    CASE
      WHEN fpd_updated_spend IS NOT NULL OR fpd_updated_impressions IS NOT NULL THEN
        CASE WHEN fpd_orig_spend IS NOT NULL THEN 'fpd_combined' ELSE 'fpd_updated_only' END
      WHEN fpd_orig_spend IS NOT NULL OR fpd_orig_impressions IS NOT NULL   THEN 'fpd_original_only'
      WHEN s_spend         IS NOT NULL OR s_impressions        IS NOT NULL  THEN
        CASE
          WHEN supplier_code IN ('FB', 'IG') THEN 'meta'
          WHEN supplier_code = 'TT'           THEN 'tiktok'
          ELSE 'social'
        END
      WHEN d_daily_recalculated_cost IS NOT NULL
        OR d_daily_recalculated_imps IS NOT NULL                            THEN 'dcm'
      ELSE 'planned_only'
    END                                                                     AS data_source_primary,

    -- ── NEW canonical video columns (did not exist in v1) ─────────────────────
    COALESCE(CAST(d_video_plays AS FLOAT64), s_video_plays)                 AS final_video_plays,
    COALESCE(CAST(d_video_comps AS FLOAT64), s_video_comps)                 AS final_video_comps

  FROM unioned u
)

-- ════════════════════════════════════════════════════════════════════════════
-- BLOCK 5: FINAL SELECT — all 130 original + 10 source-prefix + 2 new canonical = 142 cols
-- ════════════════════════════════════════════════════════════════════════════
SELECT * FROM with_rollups;


---
## Verification

Run these checks after the build to confirm the shadow table looks correct.
For full baseline comparison, run `sql/test__adif__social_mapping_v2_vs_current.sql`.

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- V1: Row count + spend/impressions by data_source_primary
SELECT
  data_source_primary,
  COUNT(*)                          AS row_count,
  MIN(date)                         AS min_date,
  MAX(date)                         AS max_date,
  ROUND(SUM(final_spend), 2)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook`
GROUP BY 1
ORDER BY total_spend DESC;

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- V2: Row count + spend/impressions by data_source_primary
SELECT
  data_source_primary,
  COUNT(*)                          AS row_count,
  MIN(date)                         AS min_date,
  MAX(date)                         AS max_date,
  ROUND(SUM(final_spend), 2)        AS total_spend,
  ROUND(SUM(final_impressions), 0)  AS total_impressions
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook_v2_test`
GROUP BY 1
ORDER BY total_spend DESC;

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- Quick source-column spot-check: confirm s_spend is populated for social rows
-- and NULL for digital rows, and d_daily_recalculated_cost is NULL for social rows
SELECT
  data_source_primary,
  COUNTIF(s_spend IS NOT NULL)                        AS rows_with_s_spend,
  COUNTIF(d_daily_recalculated_cost IS NOT NULL)      AS rows_with_dcm_cost,
  COUNTIF(fpd_spend IS NOT NULL)                      AS rows_with_fpd_spend,
  COUNTIF(updated_fpd_spend IS NOT NULL)              AS rows_with_updated_fpd,
  COUNTIF(prisma_planned_spend IS NOT NULL)           AS rows_with_prisma_plan,
  COUNTIF(pacing_planned_spend IS NOT NULL)           AS rows_with_pacing_plan,
  ROUND(SUM(s_spend), 2)                              AS total_s_spend,
  ROUND(SUM(d_daily_recalculated_cost), 2)            AS total_dcm_cost,
  ROUND(SUM(final_spend), 2)                          AS total_final_spend
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook_v2_test`
GROUP BY 1
ORDER BY 1;

In [ ]:
%%bigquery --project looker-studio-pro-452620

-- VAL-05 inline: planned_daily_spend_pk = COALESCE(prisma_planned_spend, pacing_planned_spend)
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(
    ABS(
      COALESCE(planned_daily_spend_pk, 0)
      - COALESCE(COALESCE(prisma_planned_spend, pacing_planned_spend), 0)
    ) > 0.001
  )        AS rows_with_mismatch
FROM `looker-studio-pro-452620.repo_stg.adif__mainDataTable_notebook_v2_test`;